In [2]:
# IMPORT LIBRARIES

import os
import cv2
import time
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [3]:
train_path = r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Train"
test_path = r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Test"
test_csv_path = r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Test.csv"
model_path = r"/kaggle/input/datasets/mdmafrukhosen/pretrainedgtsrb/gtsrbX.pth"

IMG_SIZE = 224
BATCH_SIZE = 32
PREPROCESSING_STRATEGY = "clahe"  # baseline | clahe | gamma | dehaze

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

basic_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Enhanced augmentations for training
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.6),  # Wider rotation
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.4),
    A.GaussNoise(std_range=(0.05, 0.15), p=0.3),
A.CoarseDropout(
    num_holes_range=(1, 1),
    hole_height_range=(32, 32),
    hole_width_range=(32, 32),
    p=0.2
),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Keep eval transforms simple
eval_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

Using device: cuda


In [5]:
# STAGE 1 — INPUT & SIMULATION
weather_transforms = A.Compose([
    A.OneOf([
        A.RandomFog(fog_coef_range=(0.2, 0.4), alpha_coef=0.1, p=1.0),
        A.RandomRain(brightness_coefficient=0.9, drop_length=5, p=1.0),
        A.RandomSnow(
    brightness_coeff=2.5,
    snow_point_range=(0.3, 0.5),
    p=1.0
),
    ], p=0.7)  # Apply one of the weather effects 70% of the time
])

def simulate_weather(image):
    return weather_transforms(image=image)["image"]

def normalize_path(path):
    return os.path.normpath(path)

def load_test_csv(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} test samples from {csv_path}")
    return df

In [9]:
# STAGE 2 — PREPROCESSING STRATEGIES

def apply_clahe(image):
    image = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(image)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    merged = cv2.merge((l, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)


def gamma_correction(image, gamma=1.5):
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(256)]).astype("uint8")
    return cv2.LUT(image, table)


def simple_dehaze(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.equalizeHist(l)
    dehazed = cv2.merge([l, a, b])
    return cv2.cvtColor(dehazed, cv2.COLOR_LAB2RGB)


def preprocess_image(image, strategy="clahe"):
    if strategy == "baseline":
        return image
    if strategy == "clahe":
        return apply_clahe(image)
    if strategy == "gamma":
        return gamma_correction(image)
    if strategy == "dehaze":
        return simple_dehaze(image)
    return image

In [10]:
class GTSRBInferenceDataset(Dataset):
    def __init__(self, csv_df, base_path, transform=None, simulate=True, strategy="clahe"):
        self.df = csv_df
        self.base_path = base_path
        self.transform = transform
        self.simulate = simulate
        self.strategy = strategy

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = normalize_path(os.path.join(self.base_path, row["Path"]))
        label = int(row["ClassId"])

        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Could not load image: {img_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.simulate:
            img = simulate_weather(img)
        img = preprocess_image(img, strategy=self.strategy)

        if self.transform:
            transformed = self.transform(image=img)
            img = transformed["image"]
        else:
            img = Image.fromarray(img)

        return img, label

In [12]:
# STAGE 1 — LOAD TEST DATASET
test_df = load_test_csv(test_csv_path)
print(f"Test CSV columns: {test_df.columns.tolist()}")
print(f"Test data shape: {test_df.shape}")

test_dataset = GTSRBInferenceDataset(
    csv_df=test_df,
    base_path=r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign",
    transform=basic_transforms,
    simulate=True,
    strategy=PREPROCESSING_STRATEGY,
)
print(f"Test dataset size: {len(test_dataset)}")

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

Loaded 12630 test samples from /kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Test.csv
Test CSV columns: ['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId', 'Path']
Test data shape: (12630, 8)
Test dataset size: 12630


In [13]:
# STAGE  — CLASSIFICATION (FROZEN MODEL)
class GTSRBModel(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        # backbone
        self.rn50 = models.resnet50(weights=None)

        # remove original fc
        in_features = 1000  # from checkpoint
        #self.rn50.fc = nn.Identity()

        # custom classifier (based on your checkpoint)
        self.dropout = nn.Dropout(0.5)  # Add dropout for regularization
        self.fl1 = nn.Linear(in_features, 256)
        self.relu = nn.ReLU()
        self.fl2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.rn50(x)
        x = self.dropout(x)  # Apply dropout
        x = self.relu(self.fl1(x))
        x = self.fl2(x)
        return x

try:
    model = GTSRBModel()
    model = model.to(device)

    checkpoint = torch.load(model_path, map_location=device)
    if "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]

    model.load_state_dict(checkpoint, strict=False)

    # Freeze all parameters, then unfreeze the last two ResNet blocks and classifier layers for fine-tuning
    for param in model.parameters():
        param.requires_grad = False
    for param in model.rn50.layer3.parameters():
        param.requires_grad = True
    for param in model.rn50.layer4.parameters():
        param.requires_grad = True
    for param in model.fl1.parameters():
        param.requires_grad = True
    for param in model.fl2.parameters():
        param.requires_grad = True

    model.eval()
    print("Custom ResNet50 model loaded and prepared for fine-tuning ✅")
except Exception as e:
    print(f"Failed to load ResNet50 model: {e}")

Custom ResNet50 model loaded and prepared for fine-tuning ✅


In [14]:
# STAGE  — FINE-TUNE THE PRETRAINED MODEL
train_csv_path = r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Train.csv"
train_df = pd.read_csv(train_csv_path)
print(f"Train CSV columns: {train_df.columns.tolist()}")
print(f"Train data shape: {train_df.shape}")

train_dataset = GTSRBInferenceDataset(
    csv_df=train_df,
    base_path=r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign",
    transform=train_transforms,
    simulate=True,
    strategy=PREPROCESSING_STRATEGY,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_dataset = GTSRBInferenceDataset(
    csv_df=test_df,
    base_path=r"/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign",
    transform=eval_transforms,
    simulate=True,
    strategy=PREPROCESSING_STRATEGY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,  # Lower LR since more layers are trainable
    weight_decay=1e-4,
)


num_epochs = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
best_val_acc = 0.0
early_stopping_patience = 5
patience_counter = 0

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * images.size(0)
        running_corrects += torch.sum(preds == labels).item()
        total += images.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = running_corrects / total

    model.eval()
    val_preds = []
    val_labels = []
    val_running_loss = 0.0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_running_loss += loss.item() * images.size(0)
            val_total += images.size(0)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss = val_running_loss / val_total
    val_acc = accuracy_score(val_labels, val_preds)
    print(
        f"Epoch {epoch}/{num_epochs} | train_loss: {epoch_loss:.4f} | train_acc: {epoch_acc * 100:.2f}% | val_loss: {val_loss:.4f} | val_acc: {val_acc * 100:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), r"/kaggle/working/gtsrbX_finetuned.pth")
        print("  Saved improved model checkpoint")
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch} epochs (no improvement for {early_stopping_patience} consecutive epochs)")
            break

print(f"Fine-tuning complete. Best validation accuracy: {best_val_acc * 100:.2f}%")


Train CSV columns: ['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId', 'Path']
Train data shape: (39209, 8)
Epoch 1/100 | train_loss: 3.0665 | train_acc: 15.11% | val_loss: 2.0667 | val_acc: 36.48%
  Saved improved model checkpoint
Epoch 2/100 | train_loss: 1.8931 | train_acc: 38.58% | val_loss: 1.4537 | val_acc: 50.22%
  Saved improved model checkpoint
Epoch 3/100 | train_loss: 1.3287 | train_acc: 56.64% | val_loss: 0.9692 | val_acc: 68.00%
  Saved improved model checkpoint
Epoch 4/100 | train_loss: 0.8636 | train_acc: 72.39% | val_loss: 0.6503 | val_acc: 78.87%
  Saved improved model checkpoint
Epoch 5/100 | train_loss: 0.6478 | train_acc: 79.43% | val_loss: 0.5078 | val_acc: 83.70%
  Saved improved model checkpoint
Epoch 6/100 | train_loss: 0.5238 | train_acc: 83.52% | val_loss: 0.5212 | val_acc: 83.19%
Epoch 7/100 | train_loss: 0.4508 | train_acc: 85.84% | val_loss: 0.4255 | val_acc: 86.00%
  Saved improved model checkpoint
Epoch 8/100 | train_loss: 0.3998 | trai